# Local Farmer Interaction Audio Processor

This notebook provides an interactive interface for processing local farmer interaction audio recordings. You can place your audio recordings in the input folder, select them from a dropdown, and run the pipeline to generate structured PDF, Excel, and Word reports in the output folder.

### Pipeline Overview:
1. **Audio Extraction/Concatenation**: Extracting/normalizing or concatenating multiple audios using FFmpeg.
2. **Transcription**: Whisper transcribe (Punjabi/Hindi/etc.).
3. **Translation**: IndicTrans2 translation to English.
4. **AI-Powered Analysis**: Gemini API extracts detailed narration, challenges, agricultural questions, rich metadata, and terminology mapping.
5. **Report Generation**: Outputting reports exactly as `reports.py` do (PDF, Excel, Word).

In [ ]:
# ---------------------------------------------------
# Google Colab & Google Drive Setup
# ---------------------------------------------------
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # 1. Mount Google Drive
    from google.colab import drive
    try:
        drive.mount('/content/drive')
        print("✅ Google Drive mounted successfully.")
    except Exception as e:
        print(f"❌ Error mounting Google Drive: {e}")
        
    # 2. Define the path to your project folder in Google Drive
    # Change this if your folder name is different
    PROJECT_PATH = "/content/drive/MyDrive/outreach_stt/App/cli-tool"
    
    import os
    import sys
    if os.path.exists(PROJECT_PATH):
        os.chdir(PROJECT_PATH)
        print(f"✅ Changed working directory to: {os.getcwd()}")
        if PROJECT_PATH not in sys.path:
            sys.path.append(PROJECT_PATH)
            
        # 3. Install packages from requirements.txt
        print("Installing dependencies from requirements.txt...")
        !pip install -r requirements.txt
        
        # 4. Install and start MongoDB inside Colab
        print("Installing and starting MongoDB...")
        !apt-get install mongodb -y > /dev/null
        !service mongodb start
        print("✅ MongoDB started successfully.")
    else:
        print(f"❌ Error: Project path '{PROJECT_PATH}' not found. Please verify the folder name in Google Drive.")
        
    # 5. Handle environment variables and API keys
    if not os.environ.get("GEMINI_API_KEY"):
        try:
            from google.colab import userdata
            os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')
            print("✅ GEMINI_API_KEY loaded from Colab Secrets.")
        except Exception:
            pass
            
    # 6. Set GPU environment variables if GPU is available
    import torch
    if torch.cuda.is_available():
        os.environ["WHISPER_DEVICE"] = "cuda"
        os.environ["TRANSLATION_DEVICE"] = "cuda"
        print("✅ GPU detected. Configured pipeline to use CUDA.")
    else:
        os.environ["WHISPER_DEVICE"] = "cpu"
        os.environ["TRANSLATION_DEVICE"] = "cpu"
        print("ℹ️ GPU not detected. Running pipeline on CPU.")

if not os.environ.get("GEMINI_API_KEY"):
    api_key = input("Enter your GEMINI_API_KEY: ").strip()
    if api_key:
        os.environ["GEMINI_API_KEY"] = api_key
        print("✅ GEMINI_API_KEY set for this session.")
    else:
        print("⚠️ Warning: GEMINI_API_KEY is not set. The pipeline might fail.")


In [ ]:
import os
import shutil
from pathlib import Path
from datetime import datetime
import asyncio
import ipywidgets as widgets
from IPython.display import display, HTML

# ---------------------------------------------------
# 1. Setup Folders
# ---------------------------------------------------
# The main folder where audios and reports will reside
MAIN_FOLDER = Path("data/local_meetings")
INPUT_DIR = MAIN_FOLDER / "input_audios"
OUTPUT_DIR = MAIN_FOLDER / "output_reports"

# Create directories if they do not exist
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Main Directory:      {MAIN_FOLDER.absolute()}")
print(f"Input Audios Folder: {INPUT_DIR.absolute()}")
print(f"Output Reports Folder: {OUTPUT_DIR.absolute()}")

In [ ]:
# ---------------------------------------------------
# 2. Import CLI Pipeline Components
# ---------------------------------------------------
import sys
# Ensure project root is in path
sys.path.append(str(Path(".").absolute()))

from src.core.database import db_manager
from src.pipeline import pipeline
from src.modules.processing import audio_extractor
from config import settings

print("✅ CLI Pipeline components imported successfully.")

In [ ]:
# ---------------------------------------------------
# 3. Directory Scanning & Concatenation Helpers
# ---------------------------------------------------
def scan_audio_files():
    """Scan input_audios/ for supported audio files"""
    supported_extensions = {'.mp3', '.wav', '.m4a', '.aac', '.ogg', '.flac', '.mp4', '.avi', '.mov', '.webm'}
    if not INPUT_DIR.exists():
        return []
    files = [f for f in INPUT_DIR.iterdir() if f.is_file() and f.suffix.lower() in supported_extensions]
    files.sort(key=lambda x: x.name)
    return files

async def copy_generated_reports(interaction_id, file_stem):
    """Locate generated reports in DB and copy them to the local output_reports/ folder"""
    record = await db_manager.get_interaction(interaction_id)
    if not record:
        raise ValueError(f"No database record found for: {interaction_id}")
    
    copied_files = []
    report_types = [
        ('excel', record.excel_report_path, '.xlsx'),
        ('pdf', record.pdf_report_path, '.pdf'),
        ('word', record.word_report_path, '.docx')
    ]
    
    for label, file_path_str, ext in report_types:
        if file_path_str and Path(file_path_str).exists():
            src_path = Path(file_path_str)
            dest_filename = f"report_{file_stem}_{interaction_id}{ext}"
            dest_path = OUTPUT_DIR / dest_filename
            shutil.copy2(src_path, dest_path)
            copied_files.append(dest_path)
            print(f"  → Copied {label.upper()} report to: {dest_path.relative_to(MAIN_FOLDER.parent)}")
            
    return copied_files

In [ ]:
# ---------------------------------------------------
# 4. Define Execution Pipeline Routines
# ---------------------------------------------------
async def process_single_audio(file_path):
    """Process a single local audio file through the pipeline"""
    metadata = {
        "date": datetime.now().strftime("%Y-%m-%d"),
        "time": datetime.now().strftime("%H:%M"),
        "village": file_path.stem,
        "block": "Unknown",
        "district": "Unknown",
        "coordinator_name": "Local Processor",
        "language": "punjabi"
    }
    
    await db_manager.connect()
    try:
        print(f"Processing audio: {file_path.name}")
        interaction_id = await pipeline.process_interaction(file_path, metadata)
        print(f"✅ Database record created: {interaction_id}")
        
        # Copy generated reports locally
        reports = await copy_generated_reports(interaction_id, file_path.stem)
        return interaction_id, reports
    except Exception as e:
        print(f"❌ Processing error: {e}")
        raise e
    finally:
        await db_manager.disconnect()

async def process_merged_audios():
    """Concatenate all audio files in input_audios/ and process the merged result"""
    audio_files = scan_audio_files()
    if not audio_files:
        print("❌ No audio files found in input_audios/ directory.")
        return None, []
        
    if len(audio_files) == 1:
        print("ℹ️ Only 1 audio file found. Processing directly without merging.")
        return await process_single_audio(audio_files[0])
        
    # Generate unique merged filename
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    merged_filename = f"merged_{timestamp}.m4a"
    merged_path = INPUT_DIR / merged_filename
    
    print(f"🔗 Concatenating {len(audio_files)} files into: {merged_path.name}...")
    await audio_extractor.concat_audio_files(audio_files, merged_path)
    print("🔗 Audio files merged successfully.")
    
    metadata = {
        "date": datetime.now().strftime("%Y-%m-%d"),
        "time": datetime.now().strftime("%H:%M"),
        "village": f"Merged_Meeting_{timestamp}",
        "block": "Unknown",
        "district": "Unknown",
        "coordinator_name": "Local Merged Processor",
        "language": "punjabi"
    }
    
    await db_manager.connect()
    try:
        print(f"Processing merged audio: {merged_path.name}")
        interaction_id = await pipeline.process_interaction(merged_path, metadata)
        print(f"✅ Database record created: {interaction_id}")
        
        # Copy generated reports locally
        reports = await copy_generated_reports(interaction_id, f"merged_{timestamp}")
        return interaction_id, reports
    except Exception as e:
        print(f"❌ Processing error: {e}")
        raise e
    finally:
        await db_manager.disconnect()

In [ ]:
# ---------------------------------------------------
# 5. Create Interactive ipywidgets UI
# ---------------------------------------------------
audio_dropdown = widgets.Dropdown(
    description='Select Audio:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='60%')
)

refresh_btn = widgets.Button(
    description='Refresh List',
    button_style='info',
    icon='refresh',
    layout=widgets.Layout(width='20%')
)

process_btn = widgets.Button(
    description='Process Selected Audio',
    button_style='success',
    icon='play',
    layout=widgets.Layout(width='39%', margin='10px 1% 10px 0')
)

merge_btn = widgets.Button(
    description='Merge & Process All',
    button_style='warning',
    icon='compress',
    layout=widgets.Layout(width='39%', margin='10px 0 10px 1%')
)

output_area = widgets.Output(
    layout=widgets.Layout(border='1px solid #ddd', padding='10px', min_height='150px')
)

def update_dropdown():
    files = scan_audio_files()
    if files:
        audio_dropdown.options = [(f.name, f) for f in files]
        audio_dropdown.disabled = False
        process_btn.disabled = False
    else:
        audio_dropdown.options = [('No audio files found in input_audios/', None)]
        audio_dropdown.disabled = True
        process_btn.disabled = True

# Callbacks
def on_refresh_clicked(b):
    with output_area:
        output_area.clear_output()
        update_dropdown()
        print("🔄 Audio directory scanned. Dropdown list updated.")

def on_process_clicked(b):
    selected_file = audio_dropdown.value
    if not selected_file:
        return
        
    process_btn.disabled = True
    merge_btn.disabled = True
    refresh_btn.disabled = True
    
    async def run_task():
        with output_area:
            output_area.clear_output()
            try:
                iid, reports = await process_single_audio(selected_file)
                print(f"\n🎉 SUCCESS! Generated reports stored in output_reports/")
            except Exception as e:
                print(f"\n❌ Processing failed: {e}")
            finally:
                process_btn.disabled = False
                merge_btn.disabled = False
                refresh_btn.disabled = False
                
    asyncio.create_task(run_task())

def on_merge_clicked(b):
    process_btn.disabled = True
    merge_btn.disabled = True
    refresh_btn.disabled = True
    
    async def run_task():
        with output_area:
            output_area.clear_output()
            try:
                iid, reports = await process_merged_audios()
                if iid:
                    print(f"\n🎉 SUCCESS! Merged reports stored in output_reports/")
            except Exception as e:
                print(f"\n❌ Processing failed: {e}")
            finally:
                process_btn.disabled = False
                merge_btn.disabled = False
                refresh_btn.disabled = False
                
    asyncio.create_task(run_task())

refresh_btn.on_click(on_refresh_clicked)
process_btn.on_click(on_process_clicked)
merge_btn.on_click(on_merge_clicked)

# Initialize UI
update_dropdown()

# Render Dashboard Layout
dashboard = widgets.VBox([
    widgets.HTML("""
    <div style='background-color: #f8f9fa; padding: 15px; border-radius: 5px; margin-bottom: 15px;'>
        <h2 style='margin-top:0; color: #2C3E50;'>Outreach Audio Processing Dashboard</h2>
        <p style='color: #7F8C8D;'>Instructions: Place your audio files (Punjabi/Hindi/English) into the <code>data/local_meetings/input_audios/</code> folder, then select an audio file to process, or merge and process all of them together.</p>
    </div>
    """),
    widgets.HBox([audio_dropdown, refresh_btn], layout=widgets.Layout(justify_content='space-between', margin='0 0 10px 0')),
    widgets.HBox([process_btn, merge_btn], layout=widgets.Layout(justify_content='center')),
    widgets.HTML("<h4>Processing Logs & Output:</h4>
"),
    output_area
])

display(dashboard)